In [1]:
!pip install rank_bm25 sentence-transformers datasets torch pandas numpy scikit-learn
# Mount Google Drive if using Colab
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch import optim
from typing import List, Tuple
from datasets import Dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
import time # Import time to measure execution time




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [2]:
PATH_COLLECTION_DATA = '/content/drive/MyDrive/Colab Notebooks/subtask4b_collection_data.pkl'
PATH_QUERY_TRAIN_DATA = '/content/drive/MyDrive/Colab Notebooks/subtask4b_query_tweets_train.tsv'
PATH_QUERY_DEV_DATA = '/content/drive/MyDrive/Colab Notebooks/subtask4b_query_tweets_dev.tsv'

# --- Load Data ---
print("Loading data...")
df_collection = pd.read_pickle(PATH_COLLECTION_DATA)
df_query_train = pd.read_csv(PATH_QUERY_TRAIN_DATA, sep='\t')
df_query_dev = pd.read_csv(PATH_QUERY_DEV_DATA, sep='\t')
print("Data loaded.")

# Display info and head of the dataframes
print("\nCollection Data Info:")
df_collection.info()
print("\nCollection Data Head:")
print(df_collection.head())

print("\nTrain Query Data Info:")
df_query_train.info()
print("\nTrain Query Data Head:")
print(df_query_train.head())

print("\nDev Query Data Info:")
df_query_dev.info()
print("\nDev Query Data Head:")
print(df_query_dev.head())

Loading data...
Data loaded.

Collection Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 7718 entries, 162 to 1056448
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   cord_uid          7718 non-null   object        
 1   source_x          7718 non-null   object        
 2   title             7718 non-null   object        
 3   doi               7677 non-null   object        
 4   pmcid             4959 non-null   object        
 5   pubmed_id         6233 non-null   object        
 6   license           7718 non-null   object        
 7   abstract          7718 non-null   object        
 8   publish_time      7715 non-null   object        
 9   authors           7674 non-null   object        
 10  journal           6668 non-null   object        
 11  mag_id            0 non-null      float64       
 12  who_covidence_id  528 non-null    object        
 13  arxiv_id          20 non-n

# Baseline

In [3]:
# Create the BM25 corpus
corpus = df_collection[:][['title', 'abstract']].apply(lambda x: f"{x['title']} {x['abstract']}", axis=1).tolist()
cord_uids = df_collection[:]['cord_uid'].tolist()
tokenized_corpus = [doc.split(' ') for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)



In [4]:
def get_top_cord_uids(query):
  text2bm25top = {}
  if query in text2bm25top.keys():
      return text2bm25top[query]
  else:
      tokenized_query = query.split(' ')
      doc_scores = bm25.get_scores(tokenized_query)
      indices = np.argsort(-doc_scores)[:5]
      bm25_topk = [cord_uids[x] for x in indices]

      text2bm25top[query] = bm25_topk
      return bm25_topk

In [5]:
# Retrieve topk candidates using the BM25 model
df_query_train['bm25_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids(x))
df_query_dev['bm25_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids(x))

# Evaluating the baseline

In [6]:
## Evaluate retrieved candidates using MRR@k
def get_performance_mrr(data, col_gold, col_pred, list_k = [1, 5, 10]):
    d_performance = {}
    for k in list_k:
        data["in_topx"] = data.apply(lambda x: (1/([i for i in x[col_pred][:k]].index(x[col_gold]) + 1) if x[col_gold] in [i for i in x[col_pred][:k]] else 0), axis=1)
        #performances.append(data["in_topx"].mean())
        d_performance[k] = data["in_topx"].mean()
    return d_performance


In [7]:
results_train = get_performance_mrr(df_query_train, 'cord_uid', 'bm25_topk')
results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'bm25_topk')
# Printed MRR@k results in the following format: {k: MRR@k}
print(f"Results on the train set: {results_train}")
print(f"Results on the dev set: {results_dev}")

Results on the train set: {1: np.float64(0.5081303975725512), 5: np.float64(0.5509777224513084), 10: np.float64(0.5509777224513084)}
Results on the dev set: {1: np.float64(0.505), 5: np.float64(0.5519166666666666), 10: np.float64(0.5519166666666666)}


# Model testing

In [8]:

merged_train = pd.merge(df_query_train, df_collection, on='cord_uid', how='inner', suffixes=('_query', '_collection'))

train_examples_cosine = []
num_negative_cosine = 1 # Number of negative examples per positive example

# Add positive examples
for _, row in merged_train.iterrows():
    # Ensure text fields are strings
    if isinstance(row['tweet_text'], str) and isinstance(row['title'], str) and isinstance(row['abstract'], str):
        train_examples_cosine.append(
            InputExample(
                texts=[row['tweet_text'], f"{row['title']}. {row['abstract']}"],
                label=1.0 # Label 1.0 for positive pair
            )
        )

# Add negative examples
# For each query, pair it with a random document that is NOT the correct one
collection_docs = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str) else "",
    axis=1
).tolist()
collection_doc_uids = df_collection['cord_uid'].tolist()

for _, positive_row in merged_train.iterrows():
    if not isinstance(positive_row['tweet_text'], str):
        continue

    # Sample negative documents
    negative_samples_df = df_collection.sample(n=num_negative_cosine * 2, replace=False) # Sample more to filter
    # Filter out the positive document
    negative_samples_df = negative_samples_df[negative_samples_df['cord_uid'] != positive_row['cord_uid']].iloc[:num_negative_cosine]

    for _, negative_row in negative_samples_df.iterrows():
         if isinstance(negative_row['title'], str) and isinstance(negative_row['abstract'], str):
            train_examples_cosine.append(
                InputExample(
                    texts=[positive_row['tweet_text'], f"{negative_row['title']}. {negative_row['abstract']}"],
                    label=0.0 # Label 0.0 for negative pair
                )
            )

print(f"\nPrepared {len(train_examples_cosine)} training examples for Cosine Similarity Loss.")


Prepared 25706 training examples for Cosine Similarity Loss.


MiniLm with Cosine SImilarity

In [9]:

merged_train = pd.merge(df_query_train, df_collection, on='cord_uid', how='inner', suffixes=('_query', '_collection'))

train_examples_cosine = []
num_negative_cosine = 1 # Number of negative examples per positive example

# Add positive examples
for _, row in merged_train.iterrows():
    # Ensure text fields are strings
    if isinstance(row['tweet_text'], str) and isinstance(row['title'], str) and isinstance(row['abstract'], str):
        train_examples_cosine.append(
            InputExample(
                texts=[row['tweet_text'], f"{row['title']}. {row['abstract']}"],
                label=1.0 # Label 1.0 for positive pair
            )
        )

# Add negative examples
# For each query, pair it with a random document that is NOT the correct one
collection_docs = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str) else "",
    axis=1
).tolist()
collection_doc_uids = df_collection['cord_uid'].tolist()

for _, positive_row in merged_train.iterrows():
    if not isinstance(positive_row['tweet_text'], str):
        continue

    negative_samples_df = df_collection.sample(n=num_negative_cosine * 2, replace=False) 
    negative_samples_df = negative_samples_df[negative_samples_df['cord_uid'] != positive_row['cord_uid']].iloc[:num_negative_cosine]

    for _, negative_row in negative_samples_df.iterrows():
         if isinstance(negative_row['title'], str) and isinstance(negative_row['abstract'], str):
            train_examples_cosine.append(
                InputExample(
                    texts=[positive_row['tweet_text'], f"{negative_row['title']}. {negative_row['abstract']}"],
                    label=0.0 # Label 0.0 for negative pair
                )
            )

print(f"\nPrepared {len(train_examples_cosine)} training examples for Cosine Similarity Loss.")


Prepared 25706 training examples for Cosine Similarity Loss.


MiniLM with Cosine Similarity Loss - Training


In [10]:

import os
os.environ["WANDB_DISABLED"] = "true"

# --- DataLoader ---
BATCH_SIZE_COSINE = 32
train_dataloader_cosine = DataLoader(train_examples_cosine, shuffle=True, batch_size=BATCH_SIZE_COSINE)


# --- Model and Loss ---
# Ensure the model is initialized if you are running this cell independently
model_cosine = SentenceTransformer('all-MiniLM-L6-v2') 
train_loss_cosine = losses.CosineSimilarityLoss(model=model_cosine) 


NUM_EPOCHS_COSINE = 3
WARMUP_STEPS = int(len(train_dataloader_cosine) * NUM_EPOCHS_COSINE * 0.1) # 10% of train data for warm-up

print(f"\nStarting training for {NUM_EPOCHS_COSINE} epochs with Cosine Similarity Loss using model.fit()...")

train_objectives = [(train_dataloader_cosine, train_loss_cosine)]

model_cosine.fit(train_objectives=train_objectives,
                  epochs=NUM_EPOCHS_COSINE,
                  optimizer_params={'lr': 2e-5},
                  warmup_steps=WARMUP_STEPS,
                  show_progress_bar=True)

print("Training finished.")

# --- Save the Fine-Tuned Model ---
model_save_path_cosine = '/content/drive/MyDrive/Colab Notebooks/minilm_cosine_tuned'
print(f"\nSaving model to {model_save_path_cosine}...")
model_cosine.save(model_save_path_cosine)
print("Model saved.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Starting training for 3 epochs with Cosine Similarity Loss using model.fit()...


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1395: FutureWarning: promote has been superseded by promote_options='default'.
  block_group = [InMemoryTable(cls._concat_blocks(list(block_group), axis=axis))]
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.079500
1000,0.060400
1500,0.052400
2000,0.046200


Training finished.

Saving model to /content/drive/MyDrive/Colab Notebooks/minilm_cosine_tuned...
Model saved.


In [11]:

def get_top_cord_uids_neural(query, trained_model, collection_embeddings, collection_cord_uids, top_k=5):
    """
    Retrieves top_k cord_uids for a given query using a trained neural model.

    Args:
        query (str): The query tweet text.
        trained_model: The fine-tuned SentenceTransformer model object.
        collection_embeddings (np.ndarray): Pre-computed embeddings for the collection.
        collection_cord_uids (list): List of cord_uids corresponding to collection_embeddings.
        top_k (int): The number of top results to retrieve.

    Returns:
        list: A list of top_k cord_uids.
    """
    if not isinstance(query, str):
        return []

    query_embedding = trained_model.encode(query, convert_to_numpy=True)

    scores = cosine_similarity(query_embedding.reshape(1, -1), collection_embeddings)[0]

    indices = np.argsort(-scores)[:top_k]

    return [collection_cord_uids[i] for i in indices]

MiniLM with Cosine Similarity Loss - Evaluation


In [12]:

print("\nPreparing and encoding collection corpus for MiniLM Cosine model...")
collection_corpus_neural = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str) else "",
    axis=1
).tolist()
collection_cord_uids_neural = df_collection['cord_uid'].tolist()

collection_embeddings_cosine = model_cosine.encode(collection_corpus_neural, show_progress_bar=True, convert_to_numpy=True)
print("Encoding complete.")


print("\nRetrieving top-k for development queries using the fine-tuned MiniLM Cosine model...")
df_query_dev['minilm_cosine_topk'] = df_query_dev['tweet_text'].apply(
    lambda x: get_top_cord_uids_neural(x, model_cosine, collection_embeddings_cosine, collection_cord_uids_neural)
)
print("Retrieval complete for dev set.")

# Evaluate performance on the dev set
print("\nEvaluating MiniLM Cosine model on the dev set...")
results_dev_minilm_cosine = get_performance_mrr(df_query_dev, 'cord_uid', 'minilm_cosine_topk')
print(f"Results on the dev set (MiniLM Cosine): {results_dev_minilm_cosine}")


Preparing and encoding collection corpus for MiniLM Cosine model...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding complete.

Retrieving top-k for development queries using the fine-tuned MiniLM Cosine model...
Retrieval complete for dev set.

Evaluating MiniLM Cosine model on the dev set...
Results on the dev set (MiniLM Cosine): {1: np.float64(0.38642857142857145), 5: np.float64(0.4546309523809524), 10: np.float64(0.4546309523809524)}


MiniLM with Multiple Negatives Ranking Loss

In [13]:

import os
os.environ["WANDB_DISABLED"] = "true"

# --- Prepare Training Examples for MultipleNegativesRankingLoss --- how='inner', suffixes=('_query', '_collection')) 

train_examples_mnr = []

for _, row in merged_train.iterrows():
    if isinstance(row['tweet_text'], str) and isinstance(row['title'], str) and isinstance(row['abstract'], str):
        train_examples_mnr.append(
            InputExample(
                texts=[row['tweet_text'], f"{row['title']}. {row['abstract']}"]
            )
        )

print(f"\nPrepared {len(train_examples_mnr)} training examples for Multiple Negatives Ranking Loss.")

BATCH_SIZE_MNR = 32
train_dataloader_mnr = DataLoader(train_examples_mnr, shuffle=True, batch_size=BATCH_SIZE_MNR)

model_mnr_minilm = SentenceTransformer('all-MiniLM-L6-v2')
train_loss_mnr_minilm = losses.MultipleNegativesRankingLoss(model=model_mnr_minilm) #

NUM_EPOCHS_MNR_MINILM = 3
WARMUP_STEPS_MNR_MINILM = int(len(train_dataloader_mnr) * NUM_EPOCHS_MNR_MINILM * 0.1) # 10% of train data for warm-up

print(f"\nStarting training for {NUM_EPOCHS_MNR_MINILM} epochs with Multiple Negatives Ranking Loss (MiniLM) using model.fit()...")

train_objectives_mnr_minilm = [(train_dataloader_mnr, train_loss_mnr_minilm)]

model_mnr_minilm.fit(train_objectives=train_objectives_mnr_minilm,
                     epochs=NUM_EPOCHS_MNR_MINILM,
                     optimizer_params={'lr': 2e-5}, # Pass optimizer learning rate
                     warmup_steps=WARMUP_STEPS_MNR_MINILM,
                     show_progress_bar=True)

print("Training finished.")

# --- Save the Fine-Tuned Model ---
model_save_path_mnr_minilm = '/content/drive/MyDrive/Colab Notebooks/minilm_mnr_tuned'
print(f"\nSaving model to {model_save_path_mnr_minilm}...")
model_mnr_minilm.save(model_save_path_mnr_minilm)
print("Model saved.")



Prepared 12853 training examples for Multiple Negatives Ranking Loss.


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



Starting training for 3 epochs with Multiple Negatives Ranking Loss (MiniLM) using model.fit()...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.401700
1000,0.257300


Training finished.

Saving model to /content/drive/MyDrive/Colab Notebooks/minilm_mnr_tuned...
Model saved.


MiniLM with Multiple Negatives Ranking Loss - Evaluation

In [14]:

print("\nPreparing and encoding collection corpus for MiniLM MNR model...")
# Use the same collection corpus and cord_uids as before
collection_corpus_neural = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str) else "",
    axis=1
).tolist()
collection_cord_uids_neural = df_collection['cord_uid'].tolist()

collection_embeddings_mnr_minilm = model_mnr_minilm.encode(collection_corpus_neural, show_progress_bar=True, convert_to_numpy=True)
print("Encoding complete.")

print("\nRetrieving top-k for development queries using the fine-tuned MiniLM MNR model...")
df_query_dev['minilm_mnr_topk'] = df_query_dev['tweet_text'].apply(
    lambda x: get_top_cord_uids_neural(x, model_mnr_minilm, collection_embeddings_mnr_minilm, collection_cord_uids_neural)
)
print("Retrieval complete for dev set.")

# Evaluate performance on the dev set
print("\nEvaluating MiniLM MNR model on the dev set...")
results_dev_minilm_mnr = get_performance_mrr(df_query_dev, 'cord_uid', 'minilm_mnr_topk')
print(f"Results on the dev set (MiniLM MNR): {results_dev_minilm_mnr}")


Preparing and encoding collection corpus for MiniLM MNR model...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding complete.

Retrieving top-k for development queries using the fine-tuned MiniLM MNR model...
Retrieval complete for dev set.

Evaluating MiniLM MNR model on the dev set...
Results on the dev set (MiniLM MNR): {1: np.float64(0.5385714285714286), 5: np.float64(0.6048809523809524), 10: np.float64(0.6048809523809524)}


MPNet with Multiple Negatives Ranking Loss

In [19]:
# -------------------------- Cell 11: Neural Experiment 3: MPNet with Multiple Negatives Ranking Loss - Data Prep and Training --------------------------

import os
os.environ["WANDB_DISABLED"] = "true"


# --- Prepare Training Examples (Same as MNR MiniLM) ---

# --- DataLoader (Can use the same as MNR MiniLM) ---
BATCH_SIZE_MPNET_MNR = 8
train_dataloader_mnr = DataLoader(train_examples_mnr, shuffle=True, batch_size=BATCH_SIZE_MPNET_MNR)
model_mnr_mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# --- Model and Loss ---
train_loss_mnr_mpnet = losses.MultipleNegativesRankingLoss(model=model_mnr_mpnet) # Ensure losses is imported.

NUM_EPOCHS_MNR_MPNET = 3 # Keep the same number of epochs initially
# Recalculate WARMUP_STEPS based on the new batch size
WARMUP_STEPS_MNR_MPNET = int(len(train_dataloader_mnr) * NUM_EPOCHS_MNR_MPNET * 0.1) # 10% of train data for warm-up

print(f"\nStarting training for {NUM_EPOCHS_MNR_MPNET} epochs with Multiple Negatives Ranking Loss (MPNet) using model.fit()...")
print(f"Using batch size: {BATCH_SIZE_MPNET_MNR}")

# Define the training objectives: a list of tuples (DataLoader, LossFunction)
train_objectives_mnr_mpnet = [(train_dataloader_mnr, train_loss_mnr_mpnet)]

# Call the fit method
model_mnr_mpnet.fit(train_objectives=train_objectives_mnr_mpnet,
                    epochs=NUM_EPOCHS_MNR_MPNET,
                    optimizer_params={'lr': 2e-5}, # Pass optimizer learning rate
                    warmup_steps=WARMUP_STEPS_MNR_MPNET,
                    show_progress_bar=True)

print("Training finished.")

# --- Save the Fine-Tuned Model ---
model_save_path_mnr_mpnet = '/content/drive/MyDrive/Colab Notebooks/mpnet_mnr_tuned'
print(f"\nSaving model to {model_save_path_mnr_mpnet}...")
model_mnr_mpnet.save(model_save_path_mnr_mpnet)
print("Model saved.")

# -------------------------- Cell 12: Neural Experiment 3: MPNet with Multiple Negatives Ranking Loss - Evaluation --------------------------
print("\nPreparing and encoding collection corpus for MPNet MNR model...")
collection_corpus_neural = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str) else "",
    axis=1
).tolist()
collection_cord_uids_neural = df_collection['cord_uid'].tolist()
collection_embeddings_mnr_mpnet = model_mnr_mpnet.encode(collection_corpus_neural, show_progress_bar=True, convert_to_numpy=True)
print("Encoding complete.")

print("\nRetrieving top-k for development queries using the fine-tuned MPNet MNR model...")
df_query_dev['mpnet_mnr_topk'] = df_query_dev['tweet_text'].apply(
    lambda x: get_top_cord_uids_neural(x, model_mnr_mpnet, collection_embeddings_mnr_mpnet, collection_cord_uids_neural)
)
print("Retrieval complete for dev set.")

# Evaluate performance on the dev set
print("\nEvaluating MPNet MNR model on the dev set...")
results_dev_mpnet_mnr = get_performance_mrr(df_query_dev, 'cord_uid', 'mpnet_mnr_topk')
print(f"Results on the dev set (MPNet MNR): {results_dev_mpnet_mnr}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.157600
1000,0.104500
1500,0.115400
2000,0.061100
2500,0.063600
3000,0.052800
3500,0.046000
4000,0.032700
4500,0.035500


Training finished.

Saving model to /content/drive/MyDrive/Colab Notebooks/mpnet_mnr_tuned...
Model saved.

Preparing and encoding collection corpus for MPNet MNR model...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding complete.

Retrieving top-k for development queries using the fine-tuned MPNet MNR model...
Retrieval complete for dev set.

Evaluating MPNet MNR model on the dev set...
Results on the dev set (MPNet MNR): {1: np.float64(0.5842857142857143), 5: np.float64(0.6521071428571429), 10: np.float64(0.6521071428571429)}


Comparison

In [20]:



print("\n--- Development Set MRR@5 Comparison ---")
print(f"MiniLM Cosine Loss: {results_dev_minilm_cosine.get(5, 'N/A'):.4f}")
print(f"MiniLM MNR Loss: {results_dev_minilm_mnr.get(5, 'N/A'):.4f}")
print(f"MPNet MNR Loss: {results_dev_mpnet_mnr.get(5, 'N/A'):.4f}")

print("\n--- Full Development Set MRR Results ---")
print("MiniLM Cosine Loss:", results_dev_minilm_cosine)
print("MiniLM MNR Loss:", results_dev_minilm_mnr)
print("MPNet MNR Loss:", results_dev_mpnet_mnr)


--- Development Set MRR@5 Comparison ---
MiniLM Cosine Loss: 0.4546
MiniLM MNR Loss: 0.6049
MPNet MNR Loss: 0.6521

--- Full Development Set MRR Results ---
MiniLM Cosine Loss: {1: np.float64(0.38642857142857145), 5: np.float64(0.4546309523809524), 10: np.float64(0.4546309523809524)}
MiniLM MNR Loss: {1: np.float64(0.5385714285714286), 5: np.float64(0.6048809523809524), 10: np.float64(0.6048809523809524)}
MPNet MNR Loss: {1: np.float64(0.5842857142857143), 5: np.float64(0.6521071428571429), 10: np.float64(0.6521071428571429)}


In [21]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os

# Define the paths to your data files and saved model
PATH_COLLECTION_DATA = '/content/drive/MyDrive/Colab Notebooks/subtask4b_collection_data.pkl'
PATH_QUERY_TEST_DATA = '/content/drive/MyDrive/Colab Notebooks/subtask4b_query_tweets_test.tsv'
MODEL_SAVE_PATH_MNR_MPNET = '/content/drive/MyDrive/Colab Notebooks/minilm_mnr_tuned'
# --- Load the trained model ---
try:
    model_mnr_mpnet = SentenceTransformer(MODEL_SAVE_PATH_MNR_MPNET)
    print(f"Successfully loaded model from {MODEL_SAVE_PATH_MNR_MPNET}")
except Exception as e:
    print(f"Error loading model: {e}")
    print(f"Please ensure the model was saved correctly to '{MODEL_SAVE_PATH_MNR_MPNET}'")
    raise


# --- Prepare Collection Data and Embeddings ---
# Load the collection data
df_collection = pd.read_pickle(PATH_COLLECTION_DATA)

# Prepare the corpus for embedding: combine title and abstract.
# Handle potential non-string values gracefully by providing an empty string fallback.
collection_corpus_neural = df_collection.apply(
    lambda row: f"{row['title']}. {row['abstract']}" if isinstance(row['title'], str) and isinstance(row['abstract'], str)
    else (row['title'] if isinstance(row['title'], str) else (row['abstract'] if isinstance(row['abstract'], str) else "")),
    axis=1
).tolist()

# Get the corresponding cord_uids for the collection
collection_cord_uids_neural = df_collection['cord_uid'].tolist()

# Encode the entire collection using the loaded model
# Set show_progress_bar=True if you want to monitor progress during encoding
print("Encoding collection corpus...")
collection_embeddings_mnr_mpnet = model_mnr_mpnet.encode(
    collection_corpus_neural,
    show_progress_bar=True, # Set to True to see progress
    convert_to_numpy=True
)
print("Encoding complete.")

# --- Load Test Data and Generate Predictions ---
print("Loading test data and generating predictions...")
df_query_test = pd.read_csv(PATH_QUERY_TEST_DATA, sep='\t')

df_query_test['neural_topk'] = df_query_test['tweet_text'].apply(
    lambda x: get_top_cord_uids_neural(x, model_mnr_mpnet, collection_embeddings_mnr_mpnet, collection_cord_uids_neural, top_k=5)
)
print("Prediction generation complete.")

# --- Format and Save Submission File ---
df_query_test['preds_list'] = df_query_test['neural_topk'].apply(lambda x: x[:5])

df_query_test['preds'] = df_query_test['preds_list'].apply(lambda x: str(x).replace(" ", "").replace('"', "'"))


# Select and save the required columns to TSV
df_submission = df_query_test[['post_id', 'preds']].copy()
submission_file_path = 'predictions.tsv'

# Save the dataframe to a TSV file without the index
df_submission.to_csv(submission_file_path, index=False, sep='\t')

print(f"Submission file created successfully at '{submission_file_path}'.")
print("Remember to zip this file to 'predictions.zip' before submitting.")


Successfully loaded model from /content/drive/MyDrive/Colab Notebooks/minilm_mnr_tuned
Encoding collection corpus...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding complete.
Loading test data and generating predictions...
Prediction generation complete.
Submission file created successfully at 'predictions.tsv'.
Remember to zip this file to 'predictions.zip' before submitting.
